# Сравнение результатов лабораторных работ 1–5

Классификация цифр и регрессия показателя прогрессирования диабета оценены на одних и тех же тестовых объектах во всех работах. Для классификации основной показатель — macro F1, выше лучше. Для регрессии — MAE, ниже лучше.

Сравниваются четыре варианта каждого алгоритма. Параметры улучшенного бейзлайна выбраны на пяти фолдах обучения sklearn. В собственной реализации эти параметры перенесены без повторного подбора. Если CV выбрала исходную конфигурацию, базовая и улучшенная версии совпадают.

Строки таблиц загружаются из JSON-файлов, записанных при выполнении ноутбуков.

In [1]:
# Читаем результаты и проверяем совпадение тестовых разбиений и полноту эксперимента.
import json
from pathlib import Path
import pandas as pd

records = []
for number in range(1, 6):
    path = Path("results") / f"lab-{number}.json"
    records.extend(json.loads(path.read_text(encoding="utf-8")))
scores = pd.DataFrame(records)
assert scores.groupby("task")["split"].nunique().eq(1).all()
assert scores.groupby(["lab", "task", "version"]).size().eq(1).all()
assert scores.groupby("task").size().to_dict() == {"classification": 20, "regression": 20}
print(scores.groupby("task")["n_test"].first().to_string())

task
classification    360
regression         89


In [2]:
# Выводим метрики всех четырёх вариантов в каждой задаче.
names = {1: "KNN", 2: "Логистическая / линейная регрессия", 3: "Дерево",
         4: "Случайный лес", 5: "Бустинг"}
versions = ["sklearn_base", "sklearn_tuned", "own_base", "own_tuned"]
scores["algorithm"] = scores["lab"].map(names)
for task, columns in (("classification", ["macro_f1", "accuracy"]),
                      ("regression", ["mae", "rmse", "r2"])):
    part = scores[scores["task"] == task]
    print(task)
    for column in columns:
        table = part.pivot(index="algorithm", columns="version", values=column)
        print(column, table.reindex(index=names.values(), columns=versions).round(4).to_string(), sep="\n")

classification
macro_f1
version                             sklearn_base  sklearn_tuned  own_base  own_tuned
algorithm                                                                           
KNN                                       0.9830         0.9830    0.9830     0.9830
Логистическая / линейная регрессия        0.9578         0.9636    0.9578     0.9636
Дерево                                    0.8230         0.8230    0.8393     0.8393
Случайный лес                             0.9606         0.9607    0.9663     0.9662
Бустинг                                   0.9325         0.9408    0.8460     0.8802
accuracy
version                             sklearn_base  sklearn_tuned  own_base  own_tuned
algorithm                                                                           
KNN                                       0.9833         0.9833    0.9833     0.9833
Логистическая / линейная регрессия        0.9583         0.9639    0.9583     0.9639
Дерево                          

In [3]:
# Определяем лучшие варианты по основной метрике и измеряем эффект настройки sklearn.
for task, key, maximize in (("classification", "macro_f1", True), ("regression", "mae", False)):
    part = scores[scores["task"] == task]
    best = part[key].max() if maximize else part[key].min()
    leaders = part[part[key].eq(best)][["algorithm", "version", key]]
    print(task, leaders.to_string(index=False), sep="\n")
    pivot = part.pivot(index="algorithm", columns="version", values=key)
    improvement = pivot["sklearn_tuned"] - pivot["sklearn_base"]
    if not maximize:
        improvement = -improvement
    print("Изменение в сторону улучшения:", improvement.round(4).to_string(), sep="\n")

classification
algorithm       version  macro_f1
      KNN  sklearn_base  0.983046
      KNN sklearn_tuned  0.983046
      KNN      own_base  0.983046
      KNN     own_tuned  0.983046
Изменение в сторону улучшения:
algorithm
KNN                                   0.0000
Бустинг                               0.0083
Дерево                                0.0000
Логистическая / линейная регрессия    0.0058
Случайный лес                         0.0001
regression
algorithm       version       mae
  Бустинг sklearn_tuned 42.328796
  Бустинг     own_tuned 42.328796
Изменение в сторону улучшения:
algorithm
KNN                                   10.9943
Бустинг                                1.2323
Дерево                                 7.9844
Логистическая / линейная регрессия    -0.0179
Случайный лес                          0.6696


## Выводы

**Распознавание цифр.** Лучший результат по MACRO_F1 — 0.9830. Его показали KNN (sklearn_base), KNN (sklearn_tuned), KNN (own_base), KNN (own_tuned). Настройка улучшила тестовый результат у 3 из пяти алгоритмов.

**Прогрессирование диабета.** Лучший результат по MAE — 42.3288. Его показали Бустинг (sklearn_tuned), Бустинг (own_tuned). Настройка улучшила тестовый результат у 4 из пяти алгоритмов.

Ранжирование описывает эти два набора и одно разбиение. Выбор лидера после просмотра общих тестовых результатов носит описательный характер. Для подтверждения преимущества нужна новая отложенная выборка или внешняя кросс-валидация. Для собственной версии классификационного бустинга отличие от sklearn включает способ обновления листьев.